# 25 · Geospatial Data Science y Spatial Machine Learning

Los datos territoriales aparecen naturalmente en gobierno, transporte, ambiente, educación, salud y servicios. El espacio rompe el supuesto iid: observaciones cercanas suelen parecerse entre sí.

## Objetivos
- Entender coordenadas, CRS y proyecciones.
- Crear geometrías y joins espaciales.
- Construir features de distancia y vecindad.
- Entender autocorrelación espacial.
- Evitar leakage por proximidad usando spatial CV.
- Introducir clustering, interpolation y modelos espaciales.


In [ ]:
!pip -q install geopandas libpysal esda
import numpy as np, pandas as pd, matplotlib.pyplot as plt, geopandas as gpd
from shapely.geometry import Point
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
SEED=42; rng=np.random.default_rng(SEED); n=1800
# nube sintética alrededor de Valparaíso/Viña para fines didácticos
lat=-33.05+rng.normal(0,.09,n); lon=-71.55+rng.normal(0,.10,n)
# target con estructura espacial + ruido
y=50+18*np.exp(-((lat+33.02)**2+(lon+71.55)**2)/.003)+6*np.sin(lon*30)+rng.normal(0,3,n)
df=pd.DataFrame({'lat':lat,'lon':lon,'target':y}); gdf=gpd.GeoDataFrame(df,geometry=gpd.points_from_xy(df.lon,df.lat),crs='EPSG:4326'); gdf.head()

## 1. CRS: lat/lon no son metros
EPSG:4326 representa coordenadas geográficas en grados. Calcular distancia Euclidiana directamente sobre lat/lon puede ser incorrecto, especialmente en áreas grandes. Para distancias/áreas proyecta a un CRS métrico apropiado o usa geodesic/haversine.

En Chile, la elección UTM depende de zona; en un proyecto real debes seleccionar CRS según ubicación.


In [ ]:
gdf.plot(column='target',figsize=(7,6),legend=True,markersize=10); plt.title('Target con patrón espacial'); plt.show()
# GeoPandas puede estimar UTM local
utm=gdf.estimate_utm_crs(); print('UTM estimado:',utm); gdf_m=gdf.to_crs(utm)

## 2. Features espaciales
Ejemplos útiles:
- distancia a hospital/colegio/estación/servicio;
- cantidad de puntos dentro de radios;
- densidad local;
- accesibilidad/tiempo de viaje;
- pertenencia a comuna/zona;
- características del vecindario;
- elevación, clima, cobertura de suelo;
- embeddings raster/satellite.

Cuidado: usar infraestructura construida **después** del periodo objetivo crea temporal leakage.


In [ ]:
# punto de referencia sintético
center=Point(gdf_m.geometry.x.median(),gdf_m.geometry.y.median()); gdf_m['dist_center_m']=gdf_m.geometry.distance(center); gdf_m[['dist_center_m','target']].corr()

## 3. Join espacial
Un spatial join asigna puntos a polígonos o relaciona geometrías por intersección/proximidad. En datos públicos, esto permite agregar registros a comuna, región, zona censal o área de servicio.

```python
points = gpd.read_file('casos.geojson')
comunas = gpd.read_file('comunas.geojson').to_crs(points.crs)
joined = gpd.sjoin(points, comunas[['comuna','geometry']], predicate='within')
```


## 4. Autocorrelación espacial
Moran's I pregunta si valores similares se agrupan espacialmente. Un valor positivo sugiere autocorrelación; un p-value por permutación ayuda a evaluar si el patrón excede lo esperado bajo aleatoriedad.


In [ ]:
from libpysal.weights import KNN
from esda.moran import Moran
w=KNN.from_dataframe(gdf_m,k=8); w.transform='r'; mi=Moran(gdf_m.target.values,w,permutations=199); print('Moran I=',mi.I,'p_perm=',mi.p_sim)

## 5. El gran problema: random split espacial
Si train y test contienen vecinos casi idénticos, un modelo puede parecer excelente porque interpola localmente. Cuando luego se despliega en otra comuna/región, cae.

Spatial CV agrupa zonas y mantiene bloques completos fuera del entrenamiento. Esto mide transferencia geográfica de forma más realista.


In [ ]:
# construir bloques geográficos simples
gdf_m['block_x']=pd.qcut(gdf_m.geometry.x,5,labels=False,duplicates='drop'); gdf_m['block_y']=pd.qcut(gdf_m.geometry.y,5,labels=False,duplicates='drop'); gdf_m['spatial_group']=gdf_m.block_x.astype(str)+'_'+gdf_m.block_y.astype(str)
X=gdf_m[['lat','lon','dist_center_m']]; y=gdf_m.target
m=RandomForestRegressor(n_estimators=350,min_samples_leaf=4,random_state=SEED,n_jobs=-1)
random=-cross_val_score(m,X,y,cv=5,scoring='neg_mean_absolute_error',n_jobs=-1).mean(); spatial=-cross_val_score(m,X,y,cv=GroupKFold(5),groups=gdf_m.spatial_group,scoring='neg_mean_absolute_error',n_jobs=-1).mean(); print('MAE random CV',random,'MAE spatial CV',spatial)

## 6. Haversine y nearest neighbors
Para puntos globales puedes usar distancia great-circle. Scikit-learn `BallTree(metric='haversine')` trabaja con coordenadas en radianes. Para redes viales, distancia Euclidiana puede ser una mala proxy: usa OpenStreetMap/OSRM/GraphHopper u otras matrices de tiempo de viaje.


In [ ]:
from sklearn.neighbors import BallTree
coords=np.deg2rad(gdf[['lat','lon']].to_numpy()); tree=BallTree(coords,metric='haversine'); dist,idx=tree.query(coords[:3],k=5); print('distancias km aprox',dist*6371)

## 7. Extensiones espaciales
- DBSCAN/HDBSCAN geográfico;
- KDE / hotspot detection;
- kriging y Gaussian Processes espaciales;
- spatial lag/error regression;
- Geographically Weighted Regression;
- raster ML y remote sensing;
- satellite imagery + CNN/ViT;
- graph neural networks sobre redes viales/territoriales;
- spatiotemporal forecasting.

## Errores comunes
- distancia en grados;
- mezclar CRS;
- random CV con fuerte autocorrelación;
- crear features de vecinos usando test;
- usar centroides sin considerar geometría;
- inferir causalidad de mapas de correlación;
- publicar ubicaciones sensibles con precisión innecesaria.

## Ejercicios
1. Compara distancia Euclidiana, Haversine y proyectada.
2. Crea polígonos sintéticos y usa `sjoin`.
3. Detecta clusters con DBSCAN usando haversine.
4. Compara random vs block CV para distintos tamaños de bloque.
5. Calcula Local Moran/LISA.
6. Construye una feature `n_servicios_2km`.
7. Descarga un dataset público geográfico y crea un mapa coroplético.
8. Investiga privacidad espacial y geomasking.
